# Gold -- KPIs de Negocio Logistico
**LogiLake | D'LOGIA** -- Capa Gold del Medallion Architecture

KPIs calculados en esta capa:
- **OTIF Rate** (On Time In Full) -- metrica principal de logistica
- **Delivery Performance** -- dias reales vs estimados, retraso promedio
- **Revenue Analytics** -- total, ticket promedio, por categoria y estado
- **Cancellation Rate** -- tasa de cancelacion mensual
- **NPS Proxy** -- distribucion de review scores

In [1]:
# SparkSession con Delta Lake 3.1.0 (PySpark 3.5.0)
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_gold")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

26/04/04 02:20:15 WARN Utils: Your hostname, JuanCamiloDev resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/04 02:20:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/devcontainers/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/devcontainers/.ivy2/cache
The jars for the packages stored in: /home/devcontainers/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1e1a05e7-21e3-4507-a0cc-c96b408deb9f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 221ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-1e1a05e7-21e3-4507-a0cc-c96b408deb9f
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/5ms)


26/04/04 02:20:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.0 | Delta Lake activo


In [2]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BASE_PATH   = "/mnt/c/logilake/data"
SILVER_PATH = f"{BASE_PATH}/silver/olist_orders"
GOLD_PATH   = f"{BASE_PATH}/gold"

os.makedirs(GOLD_PATH, exist_ok=True)

silver = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver: {silver.count():,} registros")

26/04/04 02:20:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver: 99,441 registros


In [3]:
# KPI 1: Resumen Global
kpi_global = silver.agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.col("is_delivered").cast("int")).alias("total_delivered"),
    F.sum(F.col("is_canceled").cast("int")).alias("total_canceled"),
    F.round(
        F.sum(F.when(
            (F.col("is_delivered") == True) & (F.col("is_late_delivery") == False), 1
        ).otherwise(0)).cast("double") /
        F.when(
            F.sum(F.col("is_delivered").cast("int")) > 0,
            F.sum(F.col("is_delivered").cast("int"))
        ).cast("double") * 100,
        2
    ).alias("otif_rate_pct"),
    F.round(F.avg(F.when(F.col("is_delivered") == True, F.col("delivery_days_actual"))), 1)
        .alias("avg_delivery_days_actual"),
    F.round(F.avg(F.when(F.col("is_delivered") == True, F.col("delivery_days_estimated"))), 1)
        .alias("avg_delivery_days_estimated"),
    F.round(F.avg(F.when(F.col("is_delivered") == True, F.col("delivery_delay_days"))), 1)
        .alias("avg_delay_days"),
    F.round(F.sum("payment_value"), 2).alias("total_revenue_brl"),
    F.round(F.avg("payment_value"), 2).alias("avg_order_value_brl"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    F.round(F.avg("freight_ratio") * 100, 1).alias("avg_freight_ratio_pct"),
    F.current_timestamp().alias("gold_computed_at"),
)

kpi_global.write.format("delta").mode("overwrite").save(f"/mnt/c/logilake/data/gold/kpi_global")
print("kpi_global OK")
kpi_global.show(truncate=False)

kpi_global OK


+------------+---------------+--------------+-------------+------------------------+---------------------------+--------------+-----------------+-------------------+----------------+---------------------+--------------------------+
|total_orders|total_delivered|total_canceled|otif_rate_pct|avg_delivery_days_actual|avg_delivery_days_estimated|avg_delay_days|total_revenue_brl|avg_order_value_brl|avg_review_score|avg_freight_ratio_pct|gold_computed_at          |
+------------+---------------+--------------+-------------+------------------------+---------------------------+--------------+-----------------+-------------------+----------------+---------------------+--------------------------+
|99441       |96478          |625           |93.23        |12.5                    |24.4                       |-11.9         |1.600887212E7    |160.99             |4.09            |30.8                 |2026-04-04 02:20:42.073767|
+------------+---------------+--------------+-------------+-------------

In [4]:
# KPI 2: OTIF y Delivery por Mes
kpi_monthly = (
    silver
    .withColumn("order_month", F.date_format("order_purchase_ts", "yyyy-MM"))
    .groupBy("order_month")
    .agg(
        F.count("order_id").alias("orders"),
        F.sum(F.col("is_delivered").cast("int")).alias("delivered"),
        F.sum(F.col("is_canceled").cast("int")).alias("canceled"),
        F.round(
            F.sum(F.when(
                (F.col("is_delivered") == True) & (F.col("is_late_delivery") == False), 1
            ).otherwise(0)).cast("double") /
            F.when(
                F.sum(F.col("is_delivered").cast("int")) > 0,
                F.sum(F.col("is_delivered").cast("int"))
            ).cast("double") * 100,
            2
        ).alias("otif_rate_pct"),
        F.round(F.avg(F.when(F.col("is_delivered") == True, F.col("delivery_days_actual"))), 1)
            .alias("avg_delivery_days"),
        F.round(F.avg(F.when(F.col("is_delivered") == True, F.col("delivery_delay_days"))), 1)
            .alias("avg_delay_days"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("canceled") / F.col("orders") * 100, 2))
    .orderBy("order_month")
)

kpi_monthly.write.format("delta").mode("overwrite").save(f"/mnt/c/logilake/data/gold/kpi_monthly")
print("kpi_monthly OK")
kpi_monthly.show(5)

kpi_monthly OK


+-----------+------+---------+--------+-------------+-----------------+--------------+-----------+----------------+---------------------+
|order_month|orders|delivered|canceled|otif_rate_pct|avg_delivery_days|avg_delay_days|revenue_brl|avg_review_score|cancellation_rate_pct|
+-----------+------+---------+--------+-------------+-----------------+--------------+-----------+----------------+---------------------+
|    2016-09|     4|        1|       2|          0.0|             55.0|          36.0|     252.24|             1.0|                 50.0|
|    2016-10|   324|      265|      24|        99.25|             19.6|         -36.7|   59090.48|            3.56|                 7.41|
|    2016-12|     1|        1|       0|        100.0|              5.0|         -22.0|      19.62|             5.0|                  0.0|
|    2017-01|   800|      750|       3|        97.07|             12.8|         -27.4|  138488.04|            4.06|                 0.38|
|    2017-02|  1780|     1653|    

In [5]:
# KPI 3: Revenue por Categoria
kpi_category = (
    silver
    .withColumn("category", F.explode_outer(F.col("categories")))
    .filter(F.col("category").isNotNull())
    .groupBy("category")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("payment_value"), 2).alias("avg_order_value"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
        F.round(
            F.sum(F.when(
                (F.col("is_delivered") == True) & (F.col("is_late_delivery") == False), 1
            ).otherwise(0)).cast("double") /
            F.when(
                F.sum(F.col("is_delivered").cast("int")) > 0,
                F.sum(F.col("is_delivered").cast("int"))
            ).cast("double") * 100,
            2
        ).alias("otif_rate_pct"),
    )
    .orderBy(F.desc("revenue_brl"))
)

kpi_category.write.format("delta").mode("overwrite").save(f"/mnt/c/logilake/data/gold/kpi_category")
print("kpi_category OK")
kpi_category.show(10)

kpi_category OK


+--------------------+------+-----------+---------------+----------------+-------------+
|            category|orders|revenue_brl|avg_order_value|avg_review_score|otif_rate_pct|
+--------------------+------+-----------+---------------+----------------+-------------+
|       health_beauty|  8836| 1448729.73|         163.98|            4.18|        92.49|
|       watches_gifts|  5624| 1310893.45|         233.09|            4.07|        92.61|
|      bed_bath_table|  9417| 1265918.38|         134.43|            3.97|        92.57|
|      sports_leisure|  7720|  1166060.5|         151.04|            4.17|        93.43|
|computers_accesso...|  6689| 1066263.82|         159.41|            4.03|        93.61|
|     furniture_decor|  6449|  932339.78|         144.57|            4.01|        92.88|
|          housewares|  5884|  793238.51|         134.81|            4.14|        94.64|
|          cool_stuff|  3632|  729806.15|         200.94|            4.17|         94.1|
|                auto

In [6]:
# KPI 4: Distribucion de Review Scores (NPS Proxy)
reviews_total = silver.filter(F.col("review_score").isNotNull()).count()

kpi_nps = (
    silver
    .filter(F.col("review_score").isNotNull())
    .withColumn("review_score_int", F.col("review_score").cast("integer"))
    .groupBy("review_score_int")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
        F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
    )
    .withColumn("pct_of_total",
        F.round(F.col("orders") / reviews_total * 100, 2))
    .orderBy("review_score_int")
)

kpi_nps.write.format("delta").mode("overwrite").save(f"{GOLD_PATH}/kpi_nps")
print("kpi_nps OK")
kpi_nps.show()

kpi_nps OK


+----------------+------+-----------------+-----------------+------------+
|review_score_int|orders|avg_payment_value|avg_delivery_days|pct_of_total|
+----------------+------+-----------------+-----------------+------------+
|               1| 11352|           195.83|             21.3|        11.5|
|               2|  3135|           172.76|             16.6|        3.18|
|               3|  8119|           151.74|             14.2|        8.23|
|               4| 19044|           154.81|             12.2|        19.3|
|               5| 57023|           156.33|             10.6|       57.79|
+----------------+------+-----------------+-----------------+------------+



In [7]:
# KPI 5: Delivery Performance por Estado del Vendedor
kpi_seller_state = (
    silver
    .filter(F.col("is_delivered") == True)
    .withColumn("seller_state", F.explode_outer(F.col("seller_states")))
    .filter(F.col("seller_state").isNotNull())
    .groupBy("seller_state")
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
        F.round(F.avg("delivery_delay_days"), 1).alias("avg_delay_days"),
        F.round(
            F.sum(F.when(F.col("is_late_delivery") == False, 1).otherwise(0))
            .cast("double") / F.count("order_id") * 100,
            2
        ).alias("otif_rate_pct"),
        F.round(F.sum("payment_value"), 2).alias("revenue_brl"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score"),
    )
    .orderBy(F.desc("orders"))
)

kpi_seller_state.write.format("delta").mode("overwrite").save(f"/mnt/c/logilake/data/gold/kpi_seller_state")
print("kpi_seller_state OK")
kpi_seller_state.show(10)

kpi_seller_state OK


+------------+------+-----------------+--------------+-------------+-----------+----------------+
|seller_state|orders|avg_delivery_days|avg_delay_days|otif_rate_pct|revenue_brl|avg_review_score|
+------------+------+-----------------+--------------+-------------+-----------+----------------+
|          SP| 68641|             12.3|         -11.2|        92.69|1.0017536E7|            4.12|
|          MG|  7735|             12.8|         -13.4|         95.2|  1208752.0|            4.23|
|          PR|  7512|             13.3|         -14.1|        94.65| 1448508.41|            4.18|
|          RJ|  4227|             12.0|         -12.5|        92.88|   922246.6|            4.22|
|          SC|  3603|             13.6|         -14.1|        95.14|  733185.99|            4.17|
|          RS|  1962|             11.4|         -16.3|        96.89|  434153.82|            4.32|
|          DF|   808|             12.4|         -13.3|        94.68|  113449.67|             4.1|
|          BA|   550

In [8]:
# Resumen Gold
gold_tables = [
    ("kpi_global",       "Metricas globales de toda la operacion"),
    ("kpi_monthly",      "OTIF, revenue y entrega por mes"),
    ("kpi_category",     "Revenue y OTIF por categoria de producto"),
    ("kpi_nps",          "Distribucion de review scores"),
    ("kpi_seller_state", "Performance de entrega por estado del vendedor"),
]

print("=== GOLD LAYER -- Tablas creadas ===")
for name, desc in gold_tables:
    path = f"{GOLD_PATH}/{name}"
    try:
        count = spark.read.format("delta").load(path).count()
        print(f"  OK  {name:<20} {count:>6} filas  -- {desc}")
    except Exception as e:
        print(f"  ERR {name:<20} ERROR: {e}")

=== GOLD LAYER -- Tablas creadas ===


  OK  kpi_global                1 filas  -- Metricas globales de toda la operacion


  OK  kpi_monthly              25 filas  -- OTIF, revenue y entrega por mes


  OK  kpi_category             71 filas  -- Revenue y OTIF por categoria de producto


  OK  kpi_nps                   5 filas  -- Distribucion de review scores


  OK  kpi_seller_state         22 filas  -- Performance de entrega por estado del vendedor
